# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshitha922/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I will use the action score to rank pages for human review. Pages with higher scores will appear first in the review queue. Each page will have a reason code that explains why it was selected, so the content team can understand the recommendation. The score is only a way to prioritize review; it does not guarantee that updating a page will improve its performance. A team member should review each page before deciding what action to take.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# Set up the starter repository
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

# Load the data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Recreate the ML-07 baseline score
df["score"] = np.where(
    df["impressions_90d"] >= 500,
    df["impressions_90d"],
    0
)

df["reason_code"] = np.where(
    df["score"] > 0,
    "visible",
    "not_visible"
)

df["action"] = np.where(
    df["score"] > 0,
    "REVIEW",
    "NO_ACTION"
)

df["rank"] = df["score"].rank(
    method="first",
    ascending=False
).astype(int)

baseline_queue = (
    df[["rank", "score", "reason_code", "action"]]
    .sort_values("rank")
    .reset_index(drop=True)
)

print("Rows:", len(baseline_queue))
display(baseline_queue.head(10))

Rows: 30000


,rank,score,reason_code,action
0,1,517715,visible,REVIEW
1,2,517109,visible,REVIEW
2,3,509252,visible,REVIEW
3,4,497727,visible,REVIEW
4,5,463103,visible,REVIEW
5,6,443434,visible,REVIEW
6,7,416180,visible,REVIEW
7,8,347399,visible,REVIEW
8,9,345111,visible,REVIEW
9,10,312694,visible,REVIEW


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The content and SEO teams can use this ranked queue to decide which pages to review first for a possible content refresh. The score uses search visibility to support review prioritization. It is a decision-support tool, not an automatic decision. A high score does not mean a page definitely needs an update, and a low score does not mean a page should never be reviewed. The team should check each page manually before taking action. This score does not prove that refreshing a page will improve its performance.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 2 — Intended use and limits

print("INTENDED USE")
print("Users: Content and SEO teams")
print("Purpose: Prioritize pages for manual content review.")
print("Decision: Review higher-ranked pages first.")

print("\nLIMITS")
print("The score uses search visibility only.")
print("A high score does not prove that a page needs a refresh.")
print("A low score does not mean a page should never be reviewed.")
print("The score does not prove that refreshing a page will improve performance.")

# Check the existing ranked queue
required_columns = {"rank", "score", "reason_code", "action"}
assert required_columns.issubset(baseline_queue.columns)

print("\nCheck passed: ranked queue has all required fields.")

INTENDED USE
Users: Content and SEO teams
Purpose: Prioritize pages for manual content review.
Decision: Review higher-ranked pages first.

LIMITS
The score uses search visibility only.
A high score does not prove that a page needs a refresh.
A low score does not mean a page should never be reviewed.
The score does not prove that refreshing a page will improve performance.

Check passed: ranked queue has all required fields.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before taking action, a content or SEO team member must open and review the page, check its current content, search performance, and business relevance, and decide whether an update is actually needed. The score should only help prioritize the review.

The following actions should never be automated using this score: publishing or deleting content, changing important facts, making promises about results, or deciding that a page must be refreshed without human review. A person must make the final decision and approve any changes.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 3 — Human review and no-go checks

print("HUMAN REVIEW REQUIRED")
print("A person must check the page, its content, search performance, and business relevance.")
print("A person must approve any content changes.")

print("\nNO-GO LIST")
no_go_actions = [
    "Automatically publish or delete content",
    "Automatically change important facts",
    "Promise improved performance",
    "Force a refresh without human review"
]

for item in no_go_actions:
    print("-", item)

# Check that the action queue contains only review decisions
assert set(baseline_queue["action"].unique()).issubset(
    {"REVIEW", "NO_ACTION"}
)

print("\nCheck passed: the queue does not contain automatic publishing actions.")

HUMAN REVIEW REQUIRED
A person must check the page, its content, search performance, and business relevance.
A person must approve any content changes.

NO-GO LIST
- Automatically publish or delete content
- Automatically change important facts
- Promise improved performance
- Force a refresh without human review

Check passed: the queue does not contain automatic publishing actions.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored regularly because search visibility and page performance can change over time. The team should review the score if impressions change significantly, the data becomes outdated, or the ranked pages no longer match the team's manual review findings. The model or scoring rule should be reconsidered when its recommendations become less useful or the data changes. Before using an updated rule, the team should test it again on recent data and check its results. Retraining or changing the rule should not happen automatically without review.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 4 — Monitoring and retrain triggers

print("MONITORING TRIGGERS")
triggers = [
    "Search impressions change significantly",
    "Input data becomes outdated or incomplete",
    "Ranked recommendations do not match human review",
    "The score becomes less useful for prioritizing pages"
]

for trigger in triggers:
    print("-", trigger)

print("\nREVIEW ACTION")
print("Review the scoring rule and test it on recent data before using an update.")
print("Do not retrain or change the rule automatically.")

# Basic data check
assert len(baseline_queue) > 0
assert baseline_queue["score"].notna().all()

print("\nCheck passed: the queue has rows and no missing scores.")

MONITORING TRIGGERS
- Search impressions change significantly
- Input data becomes outdated or incomplete
- Ranked recommendations do not match human review
- The score becomes less useful for prioritizing pages

REVIEW ACTION
Review the scoring rule and test it on recent data before using an update.
Do not retrain or change the rule automatically.

Check passed: the queue has rows and no missing scores.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

I will export the ranked action queue to the required output folder so it can be reused in my research paper. The export will include each page's rank, score, reason code, and action. I will check that the file is saved correctly and contains all 30,000 rows.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 5 — Export the ranked queue

import os

os.makedirs("work/outputs", exist_ok=True)

export_path = "work/outputs/content_action_playbook.csv"

# Export the ranked action queue
baseline_queue.to_csv(export_path, index=False)

# Check the exported file
exported_queue = pd.read_csv(export_path)

print("Exported file:", export_path)
print("Rows exported:", len(exported_queue))
print("Columns:", exported_queue.columns.tolist())

assert len(exported_queue) == 30000
assert exported_queue["score"].notna().all()

print("Check passed: queue exported successfully.")
display(exported_queue.head(10))

Exported file: work/outputs/content_action_playbook.csv
Rows exported: 30000
Columns: ['rank', 'score', 'reason_code', 'action']
Check passed: queue exported successfully.


,rank,score,reason_code,action
0,1,517715,visible,REVIEW
1,2,517109,visible,REVIEW
2,3,509252,visible,REVIEW
3,4,497727,visible,REVIEW
4,5,463103,visible,REVIEW
5,6,443434,visible,REVIEW
6,7,416180,visible,REVIEW
7,8,347399,visible,REVIEW
8,9,345111,visible,REVIEW
9,10,312694,visible,REVIEW


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.